# 06_clean_road_networks_compute_morphology_metrics

This notebook runs project Step 6 end to end: cleaning road networks and computing city-level and local spatial-unit morphology metrics.

**This step depends only on existing outputs from 01, 02, and 05; it does not write to 01-05 or touch 07-13.** The computation writes results city by city and network by network immediately: each `city_id x network_type` first writes to `data/06_clean_road_networks_compute_morphology_metrics/local_morphology_metric_chunks/`; final tables are assembled only after all chunks are present, so interrupted runs can resume safely.

Main definitions:

- Road-network geometry uses the GPKG `edges/nodes` layers exported by Step 02 first, avoiding repeated parsing of 50GB+ GraphML files; GraphML is still indexed and checked with a lightweight openability test.
- GPKG edges come from OSMnx directed graphs; this step first deduplicates by undirected `(min(u,v), max(u,v), key)` to avoid double-counting bidirectional road lengths.
- Length units are meters/kilometers, area units are square kilometers, and all density metrics use the effective spatial-unit area `unit_area_km2`; the compatibility acceptance field `area_km2` is also written with the same value as `unit_area_km2`.
- Local units are not dropped when empty; units with no edges or too few edges/nodes keep their rows and set `valid_metric=False`.
- Edge-length density uses exact intersection length between edge geometry and spatial units; node metrics use undirected topology degree after assigning nodes to units.
- `betweenness_gini` uses sampled Brandes betweenness approximations on the undirected city network and then computes Gini within each unit; the sample size adapts to network size so the full run remains feasible.
- `orientation_entropy` is length-weighted Shannon entropy over 36 orientation bins from 0 to 180 degrees; `orientation_order = 1 - H/log(36)`.
- `road_hierarchy_entropy` is length-weighted Shannon entropy by `highway` type; missing values are recorded as anomalies.
- `chn_003` and `chn_004` are explicitly flagged as sharing a UCDB boundary note to avoid later interpretation as fully independent boundaries.


In [ ]:
# ==== 0. Basic configuration ====
# Note: this notebook can run directly; external callers can also execute code cells sequentially with python3.
# Environment variables for debugging/resume:
# - STEP06_CITY_FILTER=main_061,chn_001  Process only selected cities
# - STEP06_NETWORK_FILTER=drive          Process only selected network types
# - STEP06_FORCE_REBUILD=1               Force rebuild of existing chunks
# - STEP06_WRITE_FINAL=0                 Skip final table assembly during debugging
# - STEP06_EDGE_CHUNK_SIZE=500000        Adjust edge chunk size

from pathlib import Path
import os
import sys
import gc
import ast
import json
import math
import time
import zlib
import traceback
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import fiona
from shapely.geometry import LineString
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False
    njit = None

ROOT = Path('/Volumes/ZHITAI2T/202606osm')
if not ROOT.exists():
    # If the notebook is moved inside the project directory, fall back to searching upward from the current working directory.
    here = Path.cwd().resolve()
    candidates = [here] + list(here.parents)
    ROOT = next((p for p in candidates if (p / 'data').exists() and (p / 'code').exists()), here)

DATA_DIR = ROOT / 'data'
CODE_DIR = ROOT / 'code_upload'
STEP_DIR = DATA_DIR / '06_clean_road_networks_compute_morphology_metrics'
TEMP_DIR = STEP_DIR / 'local_morphology_metric_chunks'
STEP_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

CITY_MASTER_PATH = DATA_DIR / '01_city_boundaries_and_sample_inventory/city_sample/city_master.parquet'
UNIT_INDEX_PATH = DATA_DIR / '05_build_multiscale_spatial_units/spatial_units_index.parquet'
UNIT_GPKG_PATHS = {
    'full_city': DATA_DIR / '05_build_multiscale_spatial_units/full_city_spatial_units.gpkg',
    'core_5km': DATA_DIR / '05_build_multiscale_spatial_units/core_5km_spatial_units.gpkg',
    'hex_1km': DATA_DIR / '05_build_multiscale_spatial_units/hex_1km_spatial_units.gpkg',
    'hex_2km': DATA_DIR / '05_build_multiscale_spatial_units/hex_2km_spatial_units.gpkg',
}
GRAPHML_DIR = DATA_DIR / '02_download_road_network_data/city_road_network_graphml'
GPKG_DIR = DATA_DIR / '02_download_road_network_data/city_road_network_gpkg'

NETWORK_TYPES = ['drive', 'walk']
CITY_SCALES = ['full_city', 'core_5km']
LOCAL_SCALES = ['hex_1km', 'hex_2km']
ALL_SCALES = CITY_SCALES + LOCAL_SCALES

CITY_FILTER = [x.strip() for x in os.environ.get('STEP06_CITY_FILTER', '').split(',') if x.strip()]
NETWORK_FILTER = [x.strip() for x in os.environ.get('STEP06_NETWORK_FILTER', '').split(',') if x.strip()]
FORCE_REBUILD = os.environ.get('STEP06_FORCE_REBUILD', '0') == '1'
WRITE_FINAL = os.environ.get('STEP06_WRITE_FINAL', '1') != '0'
EDGE_CHUNK_SIZE = int(os.environ.get('STEP06_EDGE_CHUNK_SIZE', '500000'))
MIN_VALID_EDGES = int(os.environ.get('STEP06_MIN_VALID_EDGES', '3'))
MIN_VALID_NODES = int(os.environ.get('STEP06_MIN_VALID_NODES', '3'))
ORIENTATION_BINS = 36
EPS = 1e-9

OUTPUT_CITY_CSV = STEP_DIR / 'city_morphology_metrics.csv'
OUTPUT_CITY_PARQUET = STEP_DIR / 'city_morphology_metrics.parquet'
OUTPUT_LOCAL_CSV = STEP_DIR / 'local_morphology_metrics.csv'
OUTPUT_LOCAL_PARQUET = STEP_DIR / 'local_morphology_metrics.parquet'
OUTPUT_ANOMALY_CSV = STEP_DIR / 'road_network_anomaly_records.csv'
OUTPUT_LOG_CSV = STEP_DIR / 'run_log.csv'
OUTPUT_QC_CSV = STEP_DIR / 'quality_checks.csv'
OUTPUT_INDEX_CSV = STEP_DIR / 'road_network_file_index.csv'

print('Project root:', ROOT)
print('Output directory:', STEP_DIR)
print('Python:', sys.executable)
print('Numba available:', NUMBA_AVAILABLE)
print('Debug city filter:', CITY_FILTER or 'None')
print('Debug network filter:', NETWORK_FILTER or 'None')
print('Force rebuild:', FORCE_REBUILD, 'Write final tables:', WRITE_FINAL, 'Edge chunk size:', EDGE_CHUNK_SIZE)


In [ ]:
# ==== 1. General utility functions ====

def now_iso() -> str:
    """Return a second-level ISO timestamp for sortable logs."""
    return datetime.now().isoformat(timespec='seconds')


def safe_to_csv(df: pd.DataFrame, path: Path) -> None:
    """Atomically replace a CSV via a temporary file to avoid partial final tables after interruption."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False, encoding='utf-8-sig')
    tmp.replace(path)


def safe_to_parquet(df: pd.DataFrame, path: Path) -> None:
    """Atomically replace a Parquet file via a temporary file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def append_anomaly(records: list, city_id: str, network_type: str, scale: str, anomaly_type: str,
                   message: str, severity: str = 'warning', affected_count: int | float | None = None,
                   unit_id: str | None = None, sample_value: str | None = None, source_file: str | None = None) -> None:
    """Append one standardized anomaly_records entry; large batches are aggregated with affected_count instead of logging every row."""
    records.append({
        'timestamp': now_iso(),
        'city_id': city_id,
        'network_type': network_type,
        'scale': scale,
        'unit_id': unit_id,
        'severity': severity,
        'anomaly_type': anomaly_type,
        'message': message,
        'affected_count': affected_count,
        'sample_value': sample_value,
        'source_file': source_file,
    })


def parse_city_list(master: pd.DataFrame) -> list[str]:
    """Return the cities to process after applying the debug filter."""
    cities = master['city_id'].astype(str).tolist()
    if CITY_FILTER:
        keep = set(CITY_FILTER)
        cities = [c for c in cities if c in keep]
    return cities


def parse_network_list() -> list[str]:
    """Return the network types to process after applying the debug filter."""
    nets = NETWORK_TYPES.copy()
    if NETWORK_FILTER:
        keep = set(NETWORK_FILTER)
        nets = [n for n in nets if n in keep]
    return nets


def chunk_paths(city_id: str, network_type: str) -> dict[str, Path]:
    """Return resumable file paths for each city/network pair."""
    prefix = TEMP_DIR / f'{city_id}_{network_type}'
    return {
        'city': Path(str(prefix) + '_city_morphology_metrics.parquet'),
        'local': Path(str(prefix) + '_local_morphology_metrics.parquet'),
        'anomaly': Path(str(prefix) + '_anomaly_records.csv'),
        'log': Path(str(prefix) + '_run_log.csv'),
    }


def is_chunk_complete(city_id: str, network_type: str) -> bool:
    """Return whether a city/network pair already has complete chunks."""
    p = chunk_paths(city_id, network_type)
    return p['city'].exists() and p['local'].exists() and p['log'].exists()


def normalize_highway_value(value) -> str | None:
    """Collapse the OSM highway field to one primary type; list strings use the first type."""
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, (list, tuple)):
        value = value[0] if value else None
    text = str(value).strip()
    if text == '' or text.lower() in {'nan', 'none', '<na>'}:
        return None
    if text.startswith('[') and text.endswith(']'):
        # A common Graph/GPKG form is '["residential", "service"]',fall back to simple splitting on parse failure.
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple)) and parsed:
                text = str(parsed[0]).strip()
        except Exception:
            text = text.strip('[]').split(',')[0].strip().strip('"').strip("'")
    return text.lower() if text else None


def shannon_entropy_from_weights(weights: np.ndarray) -> float:
    """Compute Shannon entropy from nonnegative weights; return NaN when weights are empty or sum to zero."""
    arr = np.asarray(weights, dtype='float64')
    arr = arr[np.isfinite(arr) & (arr > 0)]
    total = arr.sum()
    if total <= 0:
        return np.nan
    p = arr / total
    return float(-(p * np.log(p)).sum())


def gini(values: np.ndarray) -> float:
    """Compute the Gini coefficient for imbalance in within-unit betweenness approximations."""
    arr = np.asarray(values, dtype='float64')
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    if arr.size == 1:
        return 0.0
    arr = arr - arr.min() if arr.min() < 0 else arr
    total = arr.sum()
    if total <= 0:
        return 0.0
    arr.sort()
    n = arr.size
    index = np.arange(1, n + 1, dtype='float64')
    return float((2 * np.sum(index * arr) / (n * total)) - ((n + 1) / n))


def stable_seed(city_id: str, network_type: str) -> int:
    """Generate a stable random seed for sampled betweenness to ensure reproducibility."""
    return zlib.adler32(f'{city_id}-{network_type}'.encode('utf-8')) & 0xffffffff


def choose_betweenness_k(node_count: int) -> int:
    """Choose the number of Brandes sample sources adaptively by network size."""
    k_max = int(os.environ.get('STEP06_BETWEENNESS_K_MAX', '64'))
    if node_count <= 0:
        return 0
    if node_count <= 50_000:
        k = 64
    elif node_count <= 200_000:
        k = 32
    elif node_count <= 800_000:
        k = 16
    else:
        k = 8
    return min(k, k_max, node_count)


def quick_check_graphml(path: Path) -> dict:
    """Lightly check whether GraphML exists, is readable, and has node/edge tags in sampled file head/tail.

    Parsing all GraphML files directly with OSMnx is very slow for the 50GB+ file set; here a lightweight XML text check is paired with GPKG
    nodes/edges counts to complete index validation, while actual topology and geometry calculations use the matching GPKG.
    """
    result = {
        'graphml_exists': path.exists(),
        'graphml_size_mb': np.nan,
        'graphml_openable': False,
        'graphml_has_node_tag': False,
        'graphml_has_edge_tag': False,
        'graphml_check_message': '',
    }
    if not path.exists():
        result['graphml_check_message'] = 'GraphML file does not exist'
        return result
    try:
        size = path.stat().st_size
        result['graphml_size_mb'] = round(size / 1_000_000, 3)
        # Nodes are usually near the head and edges near the tail; sampling both sections is enough to identify a non-empty structure.
        with path.open('rb') as f:
            head = f.read(min(size, 4 * 1024 * 1024))
            if size > 8 * 1024 * 1024:
                f.seek(max(0, size - 8 * 1024 * 1024))
                tail = f.read(8 * 1024 * 1024)
            else:
                tail = head
        result['graphml_openable'] = True
        result['graphml_has_node_tag'] = (b'<node' in head) or (b'<node' in tail)
        result['graphml_has_edge_tag'] = (b'<edge' in head) or (b'<edge' in tail)
        if not result['graphml_has_node_tag'] or not result['graphml_has_edge_tag']:
            result['graphml_check_message'] = 'Sampled file head/tail did not contain both node and edge tags'
        else:
            result['graphml_check_message'] = 'ok'
    except Exception as exc:
        result['graphml_check_message'] = f'GraphML open failed: {type(exc).__name__}: {exc}'
    return result


In [ ]:
# ==== 2. Numba implementation of sampled betweenness ====
# This implements a sampled Brandes approximation on an unweighted undirected graph. It does not run all-source shortest paths for every node,
# but instead samples k source nodes, which makes million-node networks feasible and is sufficient for within-unit Gini comparisons.

if NUMBA_AVAILABLE:
    @njit(cache=False)
    def _approx_betweenness_unweighted(indptr, indices, sources):
        n = indptr.shape[0] - 1
        cb = np.zeros(n, dtype=np.float64)
        dist = np.empty(n, dtype=np.int32)
        sigma = np.empty(n, dtype=np.float64)
        delta = np.empty(n, dtype=np.float64)
        queue = np.empty(n, dtype=np.int64)
        stack = np.empty(n, dtype=np.int64)

        for s in sources:
            for i in range(n):
                dist[i] = -1
                sigma[i] = 0.0
                delta[i] = 0.0

            qh = 0
            qt = 0
            sp = 0
            queue[qt] = s
            qt += 1
            dist[s] = 0
            sigma[s] = 1.0

            while qh < qt:
                v = queue[qh]
                qh += 1
                stack[sp] = v
                sp += 1
                for pos in range(indptr[v], indptr[v + 1]):
                    w = indices[pos]
                    if dist[w] < 0:
                        dist[w] = dist[v] + 1
                        queue[qt] = w
                        qt += 1
                    if dist[w] == dist[v] + 1:
                        sigma[w] += sigma[v]

            for ii in range(sp - 1, -1, -1):
                w = stack[ii]
                if sigma[w] > 0.0:
                    coeff = (1.0 + delta[w]) / sigma[w]
                    for pos in range(indptr[w], indptr[w + 1]):
                        v = indices[pos]
                        if dist[v] == dist[w] - 1 and sigma[v] > 0.0:
                            delta[v] += sigma[v] * coeff
                if w != s:
                    cb[w] += delta[w]

        if sources.shape[0] > 0:
            cb = cb / sources.shape[0]
        return cb
else:
    _approx_betweenness_unweighted = None


def approximate_betweenness(csr, eligible_indices: np.ndarray, city_id: str, network_type: str) -> tuple[np.ndarray, int, str]:
    """Return sampled betweenness approximations for each node, the actual sample size, and the method description."""
    n = csr.shape[0]
    if n == 0 or eligible_indices.size == 0:
        return np.zeros(n, dtype='float64'), 0, 'empty_graph'

    k = choose_betweenness_k(int(eligible_indices.size))
    rng = np.random.default_rng(stable_seed(city_id, network_type))
    if eligible_indices.size <= k:
        sources = eligible_indices.astype(np.int64)
    else:
        sources = rng.choice(eligible_indices, size=k, replace=False).astype(np.int64)

    if NUMBA_AVAILABLE:
        values = _approx_betweenness_unweighted(csr.indptr.astype(np.int64), csr.indices.astype(np.int64), sources)
        return values, int(sources.size), 'sampled_brandes_unweighted_numba'

    # Extreme fallback: when numba is unavailable, use degree as a centrality proxy and record this explicitly in logs.
    degree_proxy = np.diff(csr.indptr).astype('float64')
    return degree_proxy, 0, 'degree_proxy_fallback_no_numba'


In [ ]:
# ==== 3. Input loading, indexing, and preflight validation ====

city_master = pd.read_parquet(CITY_MASTER_PATH)
unit_index = pd.read_parquet(UNIT_INDEX_PATH)

city_master['city_id'] = city_master['city_id'].astype(str)
unit_index['city_id'] = unit_index['city_id'].astype(str)
unit_index['scale'] = unit_index['scale'].astype(str)

expected_cities = city_master['city_id'].tolist()
scales_found = sorted(unit_index['scale'].unique().tolist())
print('city_master rows:', len(city_master), 'unique cities:', city_master['city_id'].nunique())
print('spatial_units rows:', len(unit_index), 'scale:', scales_found)
print(unit_index['scale'].value_counts().sort_index().to_string())

assert city_master['city_id'].nunique() == 86, 'city_master should cover 86 cities'
assert set(ALL_SCALES).issubset(set(scales_found)), f'spatial_units missing required scale: {ALL_SCALES}'
assert unit_index['unit_id'].nunique() == len(unit_index), 'spatial_units unit_id must be unique'

# Build the GraphML/GPKG index. GraphML gets a lightweight openability check; GPKG layer info provides non-empty node/edge evidence.
index_rows = []
preflight_anomalies = []
for city_id in expected_cities:
    for network_type in NETWORK_TYPES:
        graphml_path = GRAPHML_DIR / network_type / f'{city_id}_{network_type}.graphml'
        gpkg_path = GPKG_DIR / network_type / f'{city_id}_{network_type}.gpkg'
        row = {
            'city_id': city_id,
            'network_type': network_type,
            'graphml_path': str(graphml_path),
            'gpkg_path': str(gpkg_path),
            'gpkg_exists': gpkg_path.exists(),
            'gpkg_nodes_count': np.nan,
            'gpkg_edges_count': np.nan,
            'gpkg_crs_nodes': None,
            'gpkg_crs_edges': None,
            'gpkg_check_message': '',
        }
        row.update(quick_check_graphml(graphml_path))
        if not row['graphml_exists'] or not row['graphml_openable']:
            append_anomaly(preflight_anomalies, city_id, network_type, 'graphml', 'graphml_open_failed',
                           row['graphml_check_message'], severity='error', source_file=str(graphml_path))
        elif not row['graphml_has_node_tag'] or not row['graphml_has_edge_tag']:
            append_anomaly(preflight_anomalies, city_id, network_type, 'graphml', 'graphml_tag_incomplete',
                           row['graphml_check_message'], severity='warning', source_file=str(graphml_path))
        if not gpkg_path.exists():
            row['gpkg_check_message'] = 'GPKG file does not exist'
            append_anomaly(preflight_anomalies, city_id, network_type, 'gpkg', 'gpkg_missing',
                           row['gpkg_check_message'], severity='error', source_file=str(gpkg_path))
        else:
            try:
                layers = set(fiona.listlayers(gpkg_path))
                if not {'nodes', 'edges'}.issubset(layers):
                    row['gpkg_check_message'] = f'GPKG layers incomplete: {sorted(layers)}'
                    append_anomaly(preflight_anomalies, city_id, network_type, 'gpkg', 'gpkg_layers_missing',
                                   row['gpkg_check_message'], severity='error', source_file=str(gpkg_path))
                else:
                    info_nodes = pyogrio.read_info(gpkg_path, layer='nodes')
                    info_edges = pyogrio.read_info(gpkg_path, layer='edges')
                    row['gpkg_nodes_count'] = info_nodes.get('features')
                    row['gpkg_edges_count'] = info_edges.get('features')
                    row['gpkg_crs_nodes'] = str(info_nodes.get('crs'))
                    row['gpkg_crs_edges'] = str(info_edges.get('crs'))
                    row['gpkg_check_message'] = 'ok'
                    if not row['gpkg_nodes_count'] or not row['gpkg_edges_count']:
                        append_anomaly(preflight_anomalies, city_id, network_type, 'gpkg', 'empty_graph_gpkg',
                                       'GPKG nodes or edges are empty', severity='error', source_file=str(gpkg_path))
                    if not info_edges.get('crs') or not info_nodes.get('crs'):
                        append_anomaly(preflight_anomalies, city_id, network_type, 'gpkg', 'crs_missing',
                                       'GPKG nodes/edges CRS missing', severity='warning', source_file=str(gpkg_path))
            except Exception as exc:
                row['gpkg_check_message'] = f'GPKG check failed: {type(exc).__name__}: {exc}'
                append_anomaly(preflight_anomalies, city_id, network_type, 'gpkg', 'gpkg_open_failed',
                               row['gpkg_check_message'], severity='error', source_file=str(gpkg_path))
        index_rows.append(row)

network_index = pd.DataFrame(index_rows)
safe_to_csv(network_index, OUTPUT_INDEX_CSV)
preflight_anomaly_path = TEMP_DIR / '_preflight_anomaly_records.csv'
safe_to_csv(pd.DataFrame(preflight_anomalies), preflight_anomaly_path)

print('Road network index complete:', network_index.shape)
print(network_index.groupby('network_type')[['graphml_exists', 'graphml_openable', 'gpkg_exists']].sum().to_string())
if preflight_anomalies:
    print('Preflight anomaly count:', len(preflight_anomalies))
else:
    print('Preflight anomaly count: 0')


In [ ]:
# ==== 4. Spatial-unit, topology, and edge-chunk processing functions ====

def read_units_for_city(city_id: str, anomaly_records: list, network_type: str) -> gpd.GeoDataFrame:
    """Read four spatial-unit scales for one city and project them to the city-estimated UTM CRS."""
    parts = []
    for scale, path in UNIT_GPKG_PATHS.items():
        try:
            gdf = gpd.read_file(path, where=f"city_id = '{city_id}'")
        except Exception as exc:
            append_anomaly(anomaly_records, city_id, network_type, scale, 'unit_read_failed',
                           f'spatial_units read failed: {type(exc).__name__}: {exc}', severity='error', source_file=str(path))
            raise
        if gdf.empty:
            append_anomaly(anomaly_records, city_id, network_type, scale, 'unit_empty',
                           'spatial_units are empty for this city and scale', severity='error', source_file=str(path))
        parts.append(gdf)
    units = pd.concat(parts, ignore_index=True)
    units = gpd.GeoDataFrame(units, geometry='geometry', crs=parts[0].crs)
    if units.crs is None:
        append_anomaly(anomaly_records, city_id, network_type, 'all', 'unit_crs_missing',
                       'spatial_units CRS missing; treating as EPSG:4326', severity='warning')
        units = units.set_crs('EPSG:4326')
    invalid = (~units.geometry.is_valid).sum()
    if invalid:
        append_anomaly(anomaly_records, city_id, network_type, 'all', 'unit_invalid_geometry',
                       'spatial_units contain invalid geometry; repairing with buffer(0)', affected_count=int(invalid))
        units['geometry'] = units.geometry.buffer(0)
    try:
        target_crs = units.estimate_utm_crs()
    except Exception:
        target_crs = None
    if target_crs is None:
        target_crs = 'EPSG:3857'
        append_anomaly(anomaly_records, city_id, network_type, 'all', 'projected_crs_fallback',
                       'estimate_utm_crs failed; falling back to EPSG:3857', severity='warning')
    units = units.to_crs(target_crs)
    units['unit_code'] = np.arange(len(units), dtype='int32')
    # Use Step 05 effective area as authoritative; fall back to projected geometry area when missing or nonpositive.
    units['unit_area_km2'] = pd.to_numeric(units['unit_area_km2'], errors='coerce')
    fallback_area = units.geometry.area / 1_000_000
    bad_area = (~np.isfinite(units['unit_area_km2'])) | (units['unit_area_km2'] <= 0)
    if bad_area.any():
        units.loc[bad_area, 'unit_area_km2'] = fallback_area.loc[bad_area]
        append_anomaly(anomaly_records, city_id, network_type, 'all', 'unit_area_fallback',
                       'Some units have missing or nonpositive unit_area_km2; using projected geometry area as fallback', affected_count=int(bad_area.sum()))
    return units


def read_nodes_projected(gpkg_path: Path, target_crs, city_id: str, network_type: str, anomaly_records: list) -> gpd.GeoDataFrame:
    """Read and project the nodes layer; rebuild point geometry from x/y when missing."""
    nodes = pyogrio.read_dataframe(gpkg_path, layer='nodes', columns=['osmid', 'x', 'y'])
    nodes = gpd.GeoDataFrame(nodes, geometry='geometry', crs=nodes.crs)
    if nodes.crs is None:
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'node_crs_missing',
                       'nodes layer CRS missing; treating as EPSG:4326', severity='warning', source_file=str(gpkg_path))
        nodes = nodes.set_crs('EPSG:4326')
    missing_geom = nodes.geometry.isna() | nodes.geometry.is_empty
    if missing_geom.any():
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'node_geometry_missing',
                       'Some node geometry is missing; rebuilding from x/y', affected_count=int(missing_geom.sum()), source_file=str(gpkg_path))
        rebuilt = gpd.points_from_xy(nodes.loc[missing_geom, 'x'], nodes.loc[missing_geom, 'y'], crs=nodes.crs)
        nodes.loc[missing_geom, 'geometry'] = rebuilt
    nodes['osmid'] = pd.to_numeric(nodes['osmid'], errors='coerce')
    nodes = nodes.dropna(subset=['osmid']).copy()
    nodes['osmid'] = nodes['osmid'].astype('int64')
    nodes = nodes.drop_duplicates(subset=['osmid'], keep='first')
    nodes = nodes.to_crs(target_crs)
    nodes['x_m'] = nodes.geometry.x.astype('float64')
    nodes['y_m'] = nodes.geometry.y.astype('float64')
    return nodes


def build_topology(edge_uv: pd.DataFrame, nodes: gpd.GeoDataFrame, city_id: str, network_type: str,
                   anomaly_records: list, source_file: str) -> dict:
    """Build undirected deduplicated topology, degree, component, and betweenness from GPKG edge u/v/key fields."""
    t0 = time.time()
    total_edges_raw = len(edge_uv)
    edge_uv = edge_uv.copy()
    edge_uv['row_id'] = np.arange(total_edges_raw, dtype='int64')
    for col in ['u', 'v']:
        edge_uv[col] = pd.to_numeric(edge_uv[col], errors='coerce')
    if 'key' not in edge_uv.columns:
        edge_uv['key'] = 0
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_key_missing',
                       'edges layer is missing the key field; treating it as 0', severity='warning', source_file=source_file)
    edge_uv['key'] = edge_uv['key'].fillna(0).astype(str)
    bad_uv = edge_uv['u'].isna() | edge_uv['v'].isna()
    if bad_uv.any():
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_endpoint_missing',
                       'Some edges are missing u/v endpoints and were excluded from topology calculation', affected_count=int(bad_uv.sum()), source_file=source_file)
    edge_uv = edge_uv.loc[~bad_uv].copy()
    edge_uv['u'] = edge_uv['u'].astype('int64')
    edge_uv['v'] = edge_uv['v'].astype('int64')
    edge_uv['a'] = np.minimum(edge_uv['u'].to_numpy(), edge_uv['v'].to_numpy())
    edge_uv['b'] = np.maximum(edge_uv['u'].to_numpy(), edge_uv['v'].to_numpy())

    dedup_local = ~edge_uv.duplicated(subset=['a', 'b', 'key'], keep='first')
    keep_row_ids = edge_uv.loc[dedup_local, 'row_id'].to_numpy(dtype='int64')
    keep_mask = np.zeros(total_edges_raw, dtype=bool)
    keep_mask[keep_row_ids] = True
    unique_edges = edge_uv.loc[dedup_local].copy()
    duplicate_count = total_edges_raw - len(unique_edges)
    if duplicate_count > 0:
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'directed_duplicate_edges_removed',
                       'Removed directed reverse/duplicate edges by undirected (min(u,v), max(u,v), key) to avoid duplicate length',
                       severity='info', affected_count=int(duplicate_count), source_file=source_file)

    node_ids = nodes['osmid'].to_numpy(dtype='int64')
    node_index = pd.Series(np.arange(len(nodes), dtype='int64'), index=node_ids)

    graph_edges = unique_edges.loc[unique_edges['a'] != unique_edges['b'], ['a', 'b', 'row_id']].copy()
    ui = graph_edges['a'].map(node_index)
    vi = graph_edges['b'].map(node_index)
    valid_pair = ui.notna() & vi.notna()
    missing_nodes = int((~valid_pair).sum())
    if missing_nodes:
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_endpoint_node_not_found',
                       'Some edge endpoints are absent from the nodes layer and were excluded from component/degree calculation', affected_count=missing_nodes,
                       source_file=source_file)
    ui = ui.loc[valid_pair].astype('int64').to_numpy()
    vi = vi.loc[valid_pair].astype('int64').to_numpy()
    graph_row_ids = graph_edges.loc[valid_pair, 'row_id'].to_numpy(dtype='int64')

    n_nodes = len(nodes)
    if len(ui) > 0:
        rows = np.concatenate([ui, vi])
        cols = np.concatenate([vi, ui])
        data = np.ones(len(rows), dtype=np.uint8)
        csr = coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes)).tocsr()
        csr.data[:] = 1
        n_components, labels = connected_components(csr, directed=False, return_labels=True)
        degree_arr = np.diff(csr.indptr).astype('int32')
        eligible = np.flatnonzero(degree_arr > 0).astype('int64')
        betw_arr, betw_k, betw_method = approximate_betweenness(csr, eligible, city_id, network_type)
    else:
        csr = coo_matrix((n_nodes, n_nodes)).tocsr()
        n_components = n_nodes
        labels = np.arange(n_nodes, dtype='int32')
        degree_arr = np.zeros(n_nodes, dtype='int32')
        betw_arr = np.zeros(n_nodes, dtype='float64')
        betw_k = 0
        betw_method = 'empty_graph'

    row_component = np.full(total_edges_raw, -1, dtype='int32')
    if len(graph_row_ids) > 0:
        row_component[graph_row_ids] = labels[ui].astype('int32')

    nodes = nodes.copy()
    nodes['degree'] = degree_arr
    nodes['betweenness_approx'] = betw_arr.astype('float64')
    nodes['component_id'] = labels.astype('int32')

    return {
        'nodes': nodes,
        'keep_mask': keep_mask,
        'row_component': row_component,
        'total_edges_raw': int(total_edges_raw),
        'unique_edges_count': int(len(unique_edges)),
        'duplicate_edges_removed': int(duplicate_count),
        'component_count_graph': int(n_components),
        'betweenness_k': int(betw_k),
        'betweenness_method': betw_method,
        'topology_seconds': round(time.time() - t0, 3),
    }


def orientation_bins_from_xy(ux, uy, vx, vy) -> np.ndarray:
    """Compute 0-180 degree undirected orientation bins from edge endpoint coordinates."""
    dx = vx - ux
    dy = vy - uy
    valid = np.isfinite(dx) & np.isfinite(dy) & ((np.abs(dx) + np.abs(dy)) > EPS)
    angle = (np.degrees(np.arctan2(dx, dy)) % 180.0)
    bins = np.full(len(angle), -1, dtype='int16')
    bins[valid] = np.floor(angle[valid] / (180.0 / ORIENTATION_BINS)).astype('int16')
    bins[bins >= ORIENTATION_BINS] = ORIENTATION_BINS - 1
    return bins


def rebuild_missing_edge_geometries(edges: gpd.GeoDataFrame, node_x: pd.Series, node_y: pd.Series,
                                    city_id: str, network_type: str, anomaly_records: list, source_file: str) -> gpd.GeoDataFrame:
    """Fill missing edge geometry with endpoint-coordinate LineStrings; normal data should rarely enter this branch."""
    missing = edges.geometry.isna() | edges.geometry.is_empty
    if not missing.any():
        return edges
    append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_geometry_missing',
                   'Some edge geometry is missing; attempting to fill lines from node coordinates', affected_count=int(missing.sum()), source_file=source_file)
    rebuilt = []
    for _, row in edges.loc[missing, ['u', 'v']].iterrows():
        ux = node_x.get(row['u'], np.nan)
        uy = node_y.get(row['u'], np.nan)
        vx = node_x.get(row['v'], np.nan)
        vy = node_y.get(row['v'], np.nan)
        if np.isfinite([ux, uy, vx, vy]).all():
            rebuilt.append(LineString([(ux, uy), (vx, vy)]))
        else:
            rebuilt.append(None)
    edges.loc[missing, 'geometry'] = rebuilt
    still_missing = (edges.geometry.isna() | edges.geometry.is_empty).sum()
    if still_missing:
        append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_geometry_rebuild_failed',
                       'Some missing edges could not be filled from node coordinates', affected_count=int(still_missing), source_file=source_file)
    return edges


In [ ]:
# ==== 5. Metric aggregation functions ====

def aggregate_orientation_entropy(edge_units: pd.DataFrame, unit_count: int) -> tuple[np.ndarray, np.ndarray]:
    """Compute length-weighted orientation entropy and orientation order by unit."""
    entropy = np.full(unit_count, np.nan, dtype='float64')
    order = np.full(unit_count, np.nan, dtype='float64')
    valid = edge_units[(edge_units['orientation_bin'] >= 0) & np.isfinite(edge_units['clip_len_m']) & (edge_units['clip_len_m'] > 0)]
    if valid.empty:
        return entropy, order
    hist = valid.groupby(['unit_code', 'orientation_bin'], observed=True)['clip_len_m'].sum().reset_index()
    for unit_code, group in hist.groupby('unit_code', sort=False):
        h = shannon_entropy_from_weights(group['clip_len_m'].to_numpy())
        entropy[int(unit_code)] = h
        if np.isfinite(h):
            order[int(unit_code)] = 1.0 - (h / math.log(ORIENTATION_BINS))
    return entropy, order


def aggregate_highway_entropy(edge_units: pd.DataFrame, unit_count: int) -> np.ndarray:
    """Compute length-weighted Shannon entropy of highway type by unit."""
    entropy = np.full(unit_count, np.nan, dtype='float64')
    valid = edge_units[(edge_units['highway_code'] >= 0) & np.isfinite(edge_units['clip_len_m']) & (edge_units['clip_len_m'] > 0)]
    if valid.empty:
        return entropy
    hist = valid.groupby(['unit_code', 'highway_code'], observed=True)['clip_len_m'].sum().reset_index()
    for unit_code, group in hist.groupby('unit_code', sort=False):
        entropy[int(unit_code)] = shannon_entropy_from_weights(group['clip_len_m'].to_numpy())
    return entropy


def aggregate_component_metrics(edge_units: pd.DataFrame, unit_count: int) -> tuple[np.ndarray, np.ndarray]:
    """Use full-city undirected component labels to compute within-unit component count and largest-component edge share."""
    component_count = np.full(unit_count, np.nan, dtype='float64')
    giant_share = np.full(unit_count, np.nan, dtype='float64')
    valid = edge_units[edge_units['component_id'] >= 0]
    if valid.empty:
        return component_count, giant_share
    counts = valid.groupby(['unit_code', 'component_id'], observed=True).size().rename('n').reset_index()
    total = counts.groupby('unit_code')['n'].sum()
    max_n = counts.groupby('unit_code')['n'].max()
    n_comp = counts.groupby('unit_code')['component_id'].nunique()
    idx = total.index.astype('int64').to_numpy()
    component_count[idx] = n_comp.reindex(total.index).to_numpy(dtype='float64')
    giant_share[idx] = (max_n / total).reindex(total.index).to_numpy(dtype='float64')
    return component_count, giant_share


def build_metrics_table(units: gpd.GeoDataFrame, nodes: gpd.GeoDataFrame, edge_units: pd.DataFrame,
                        city_info: pd.Series, network_type: str, anomaly_records: list) -> pd.DataFrame:
    """Merge node aggregates, edge aggregates, and spatial-unit metadata into the final metric table."""
    unit_count = len(units)
    unit_meta_cols = [
        'unit_code', 'unit_id', 'city_id', 'city_name_en', 'scale', 'sample_group', 'unit_area_km2',
        'city_area_km2', 'valid_area_ratio', 'edge_unit', 'country', 'iso3', 'region',
        'center_lon', 'center_lat', 'population_weight', 'built_share', 'population_sum'
    ]
    for col in unit_meta_cols:
        if col not in units.columns:
            units[col] = np.nan
    metrics = pd.DataFrame(units[unit_meta_cols].drop(columns=[], errors='ignore'))
    # The acceptance checklist requires area_km2; keep unit_area_km2 for upstream compatibility, with identical values.
    metrics['area_km2'] = metrics['unit_area_km2']
    metrics['network_type'] = network_type
    metrics['shared_ucdb_boundary_note'] = np.where(
        metrics['city_id'].isin(['chn_003', 'chn_004']),
        'chn_003 and chn_004 use the same UCDB boundary candidate; downstream interpretation should not treat them as fully independent boundaries',
        ''
    )

    # Initialize metric fields. Counts/lengths default to 0 and distribution metrics default to NaN.
    zero_cols = ['node_count', 'edge_count', 'total_edge_length_m', 'total_edge_length_km',
                 'edge_density_km_per_km2', 'node_density_per_km2', 'intersection_density_per_km2']
    nan_cols = ['average_node_degree', 'four_way_share', 'dead_end_share',
                'segment_length_mean', 'segment_length_median', 'segment_length_p90',
                'edge_circuity_mean', 'edge_circuity_median',
                'orientation_entropy', 'orientation_order', 'component_count',
                'giant_component_edge_share', 'betweenness_gini', 'road_hierarchy_entropy']
    for col in zero_cols:
        metrics[col] = 0.0
    for col in nan_cols:
        metrics[col] = np.nan

    # Assign nodes to units: degree uses undirected deduplicated topology, with dead_end/four_way defined as degree=1 and degree>=4 as requested.
    node_cols = ['osmid', 'degree', 'betweenness_approx', 'geometry']
    node_join = gpd.sjoin(nodes[node_cols], units[['unit_code', 'geometry']], how='inner', predicate='within')
    if not node_join.empty:
        ng = node_join.groupby('unit_code', observed=True)
        node_count = ng.size()
        avg_degree = ng['degree'].mean()
        dead_share = ng['degree'].apply(lambda s: float((s == 1).sum()) / len(s) if len(s) else np.nan)
        four_share = ng['degree'].apply(lambda s: float((s >= 4).sum()) / len(s) if len(s) else np.nan)
        intersection_count = ng['degree'].apply(lambda s: int((s >= 3).sum()))
        betw_gini = ng['betweenness_approx'].apply(lambda s: gini(s.to_numpy()))
        idx = node_count.index.astype('int64').to_numpy()
        metrics.loc[idx, 'node_count'] = node_count.to_numpy(dtype='float64')
        metrics.loc[idx, 'average_node_degree'] = avg_degree.reindex(node_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'dead_end_share'] = dead_share.reindex(node_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'four_way_share'] = four_share.reindex(node_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'intersection_density_per_km2'] = intersection_count.reindex(node_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'betweenness_gini'] = betw_gini.reindex(node_count.index).to_numpy(dtype='float64')

    # Assign edges to units: length uses precisely clipped line-segment length, and distribution metrics use edge segments within units.
    if not edge_units.empty:
        eg = edge_units.groupby('unit_code', observed=True)
        edge_count = eg.size()
        total_len = eg['clip_len_m'].sum()
        seg_mean = eg['clip_len_m'].mean()
        seg_median = eg['clip_len_m'].quantile(0.5)
        seg_p90 = eg['clip_len_m'].quantile(0.9)
        circ_mean = eg['edge_circuity'].mean()
        circ_median = eg['edge_circuity'].quantile(0.5)
        idx = edge_count.index.astype('int64').to_numpy()
        metrics.loc[idx, 'edge_count'] = edge_count.to_numpy(dtype='float64')
        metrics.loc[idx, 'total_edge_length_m'] = total_len.reindex(edge_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'segment_length_mean'] = seg_mean.reindex(edge_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'segment_length_median'] = seg_median.reindex(edge_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'segment_length_p90'] = seg_p90.reindex(edge_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'edge_circuity_mean'] = circ_mean.reindex(edge_count.index).to_numpy(dtype='float64')
        metrics.loc[idx, 'edge_circuity_median'] = circ_median.reindex(edge_count.index).to_numpy(dtype='float64')

        orientation_entropy, orientation_order = aggregate_orientation_entropy(edge_units, unit_count)
        hierarchy_entropy = aggregate_highway_entropy(edge_units, unit_count)
        component_count, giant_share = aggregate_component_metrics(edge_units, unit_count)
        metrics['orientation_entropy'] = orientation_entropy
        metrics['orientation_order'] = orientation_order
        metrics['road_hierarchy_entropy'] = hierarchy_entropy
        metrics['component_count'] = component_count
        metrics['giant_component_edge_share'] = giant_share

    # Density and validity.
    metrics['total_edge_length_km'] = metrics['total_edge_length_m'] / 1000.0
    area = pd.to_numeric(metrics['unit_area_km2'], errors='coerce')
    good_area = np.isfinite(area) & (area > 0)
    metrics.loc[good_area, 'edge_density_km_per_km2'] = metrics.loc[good_area, 'total_edge_length_km'] / area.loc[good_area]
    metrics.loc[good_area, 'node_density_per_km2'] = metrics.loc[good_area, 'node_count'] / area.loc[good_area]
    # intersection_density temporarily stores intersection_count above; divide by area here.
    metrics.loc[good_area, 'intersection_density_per_km2'] = metrics.loc[good_area, 'intersection_density_per_km2'] / area.loc[good_area]

    metrics['valid_metric'] = (
        good_area
        & (metrics['edge_count'] >= MIN_VALID_EDGES)
        & (metrics['node_count'] >= MIN_VALID_NODES)
    )
    metrics['validity_note'] = ''
    metrics.loc[~good_area, 'validity_note'] = 'area_invalid'
    metrics.loc[good_area & (metrics['edge_count'] == 0), 'validity_note'] = 'empty_unit_no_edges'
    metrics.loc[good_area & (metrics['edge_count'] > 0) & (metrics['edge_count'] < MIN_VALID_EDGES), 'validity_note'] = 'too_few_edges'
    metrics.loc[good_area & (metrics['node_count'] < MIN_VALID_NODES), 'validity_note'] = np.where(
        metrics.loc[good_area & (metrics['node_count'] < MIN_VALID_NODES), 'validity_note'].eq(''),
        'too_few_nodes',
        metrics.loc[good_area & (metrics['node_count'] < MIN_VALID_NODES), 'validity_note'] + ';too_few_nodes'
    )
    metrics['edge_assignment_method'] = 'exact_line_polygon_intersection_length'
    metrics['degree_definition'] = 'undirected_unique_neighbor_degree_after_reverse_edge_dedup'
    metrics['betweenness_method'] = 'sampled_brandes_unweighted_city_graph_then_unit_gini'

    numeric_cols = zero_cols + nan_cols
    inf_mask = np.isinf(metrics[numeric_cols]).any(axis=1)
    if inf_mask.any():
        anomaly_city_id = str(metrics['city_id'].iloc[0]) if 'city_id' in metrics.columns and len(metrics) else str(getattr(city_info, 'name', ''))
        for scale, cnt in metrics.loc[inf_mask].groupby('scale').size().items():
            append_anomaly(anomaly_records, anomaly_city_id, network_type, scale, 'nan_or_inf_metric',
                           'Metrics contained inf values, which were replaced with NaN', affected_count=int(cnt))
        metrics[numeric_cols] = metrics[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # Aggregate anomaly records for empty/sparse units by scale to avoid expanding the anomaly table to hundreds of thousands of rows.
    anomaly_city_id = str(metrics['city_id'].iloc[0]) if 'city_id' in metrics.columns and len(metrics) else str(getattr(city_info, 'name', ''))
    for scale, sub in metrics.groupby('scale', sort=False):
        empty_count = int((sub['edge_count'] == 0).sum())
        invalid_count = int((~sub['valid_metric']).sum())
        if empty_count:
            append_anomaly(anomaly_records, anomaly_city_id, network_type, scale, 'empty_units',
                           'This scale has units with no edges; rows were retained with valid_metric=False', affected_count=empty_count,
                           sample_value=','.join(sub.loc[sub['edge_count'] == 0, 'unit_id'].astype(str).head(5)))
        if invalid_count:
            append_anomaly(anomaly_records, anomaly_city_id, network_type, scale, 'invalid_metric_units',
                           f'This scale has units with edge count < {MIN_VALID_EDGES} or node count < {MIN_VALID_NODES} units',
                           affected_count=invalid_count,
                           sample_value=','.join(sub.loc[~sub['valid_metric'], 'unit_id'].astype(str).head(5)))

    ordered = [
        'city_id', 'city_name_en', 'country', 'iso3', 'region', 'sample_group', 'network_type',
        'scale', 'unit_id', 'unit_area_km2', 'area_km2', 'city_area_km2', 'valid_area_ratio', 'edge_unit',
        'center_lon', 'center_lat', 'population_weight', 'built_share', 'population_sum',
        'valid_metric', 'validity_note',
        'node_count', 'edge_count', 'total_edge_length_m', 'total_edge_length_km',
        'edge_density_km_per_km2', 'node_density_per_km2', 'intersection_density_per_km2',
        'average_node_degree', 'four_way_share', 'dead_end_share',
        'segment_length_mean', 'segment_length_median', 'segment_length_p90',
        'edge_circuity_mean', 'edge_circuity_median',
        'orientation_entropy', 'orientation_order', 'component_count', 'giant_component_edge_share',
        'betweenness_gini', 'road_hierarchy_entropy',
        'edge_assignment_method', 'degree_definition', 'betweenness_method', 'shared_ucdb_boundary_note'
    ]
    for col in ordered:
        if col not in metrics.columns:
            metrics[col] = np.nan
    return metrics[ordered].copy()


In [ ]:
# ==== 6. Complete calculation for one city/network pair ====

def process_city_network(city_id: str, network_type: str, city_info: pd.Series) -> None:
    """Compute all scale metrics for one city/network pair and immediately write chunks, logs, and anomalies."""
    paths = chunk_paths(city_id, network_type)
    if is_chunk_complete(city_id, network_type) and not FORCE_REBUILD:
        print(f'[{now_iso()}] Skipping existing chunk: {city_id} {network_type}')
        return

    t_all = time.time()
    anomaly_records = []
    log = {
        'timestamp': now_iso(),
        'city_id': city_id,
        'city_name_en': city_info.get('city_name_en', ''),
        'network_type': network_type,
        'status': 'started',
        'message': '',
        'raw_edge_count': np.nan,
        'unique_edge_count': np.nan,
        'duplicate_edges_removed': np.nan,
        'node_count': np.nan,
        'unit_count': np.nan,
        'city_metric_rows': np.nan,
        'local_metric_rows': np.nan,
        'component_count_graph': np.nan,
        'betweenness_k': np.nan,
        'betweenness_method': '',
        'edge_candidate_rows': np.nan,
        'edge_clip_zero_count': np.nan,
        'circuity_invalid_count': np.nan,
        'orientation_invalid_count': np.nan,
        'highway_missing_count': np.nan,
        'seconds_total': np.nan,
        'seconds_topology': np.nan,
        'seconds_edge_overlay': np.nan,
        'source_gpkg': '',
    }

    gpkg_path = GPKG_DIR / network_type / f'{city_id}_{network_type}.gpkg'
    log['source_gpkg'] = str(gpkg_path)
    try:
        if not gpkg_path.exists():
            raise FileNotFoundError(f'Missing GPKG: {gpkg_path}')

        print(f'[{now_iso()}] Starting {city_id} {network_type}')
        units = read_units_for_city(city_id, anomaly_records, network_type)
        log['unit_count'] = len(units)
        target_crs = units.crs
        units_geom = units.set_index('unit_code').geometry

        nodes = read_nodes_projected(gpkg_path, target_crs, city_id, network_type, anomaly_records)
        log['node_count'] = len(nodes)
        node_x = pd.Series(nodes['x_m'].to_numpy(), index=nodes['osmid'].to_numpy())
        node_y = pd.Series(nodes['y_m'].to_numpy(), index=nodes['osmid'].to_numpy())

        # First read only u/v/key to build undirected deduplicated topology, components, and betweenness.
        edge_uv = pyogrio.read_dataframe(gpkg_path, layer='edges', columns=['u', 'v', 'key'], read_geometry=False)
        topology = build_topology(edge_uv, nodes, city_id, network_type, anomaly_records, str(gpkg_path))
        nodes = topology['nodes']
        keep_mask = topology['keep_mask']
        row_component = topology['row_component']
        log['raw_edge_count'] = topology['total_edges_raw']
        log['unique_edge_count'] = topology['unique_edges_count']
        log['duplicate_edges_removed'] = topology['duplicate_edges_removed']
        log['component_count_graph'] = topology['component_count_graph']
        log['betweenness_k'] = topology['betweenness_k']
        log['betweenness_method'] = topology['betweenness_method']
        log['seconds_topology'] = topology['topology_seconds']
        del edge_uv
        gc.collect()

        highway_to_code = {}
        edge_unit_parts = []
        edge_candidate_rows = 0
        edge_clip_zero_count = 0
        circuity_invalid_count = 0
        orientation_invalid_count = 0
        highway_missing_count = 0
        overlay_start = time.time()

        total_features = int(log['raw_edge_count'])
        for start in range(0, total_features, EDGE_CHUNK_SIZE):
            max_features = min(EDGE_CHUNK_SIZE, total_features - start)
            edges = pyogrio.read_dataframe(
                gpkg_path,
                layer='edges',
                columns=['u', 'v', 'key', 'length', 'highway'],
                skip_features=start,
                max_features=max_features,
            )
            if edges.empty:
                continue
            edges = gpd.GeoDataFrame(edges, geometry='geometry', crs=edges.crs)
            if edges.crs is None:
                append_anomaly(anomaly_records, city_id, network_type, 'graph', 'edge_crs_missing',
                               'edges layer CRS missing; treating as EPSG:4326', severity='warning', source_file=str(gpkg_path))
                edges = edges.set_crs('EPSG:4326')
            edges['row_id'] = np.arange(start, start + len(edges), dtype='int64')
            # Keep the first edge after undirected deduplication to avoid double-counting bidirectional road length.
            edges = edges.loc[keep_mask[edges['row_id'].to_numpy()]].copy()
            if edges.empty:
                continue
            for col in ['u', 'v']:
                edges[col] = pd.to_numeric(edges[col], errors='coerce')
            edges = edges.dropna(subset=['u', 'v']).copy()
            edges['u'] = edges['u'].astype('int64')
            edges['v'] = edges['v'].astype('int64')
            edges = edges.to_crs(target_crs)
            edges = rebuild_missing_edge_geometries(edges, node_x, node_y, city_id, network_type, anomaly_records, str(gpkg_path))
            edges = edges.loc[~(edges.geometry.isna() | edges.geometry.is_empty)].copy()
            if edges.empty:
                continue

            geom_len = edges.geometry.length.to_numpy(dtype='float64')
            ux = edges['u'].map(node_x).to_numpy(dtype='float64')
            uy = edges['u'].map(node_y).to_numpy(dtype='float64')
            vx = edges['v'].map(node_x).to_numpy(dtype='float64')
            vy = edges['v'].map(node_y).to_numpy(dtype='float64')
            straight = np.sqrt((vx - ux) ** 2 + (vy - uy) ** 2)
            circuity = np.full(len(edges), np.nan, dtype='float32')
            good_circ = np.isfinite(geom_len) & np.isfinite(straight) & (straight > 0.01) & (geom_len > 0)
            circuity[good_circ] = (geom_len[good_circ] / straight[good_circ]).astype('float32')
            # Clamp values slightly below 1 from projection/rounding back to 1 to avoid an invalid circuity lower bound.
            circuity[(circuity > 0) & (circuity < 1)] = 1.0
            invalid_c = int((~good_circ).sum())
            circuity_invalid_count += invalid_c

            orientation_bin = orientation_bins_from_xy(ux, uy, vx, vy)
            orientation_invalid_count += int((orientation_bin < 0).sum())

            highway_class = edges['highway'].map(normalize_highway_value) if 'highway' in edges.columns else pd.Series([None] * len(edges), index=edges.index)
            missing_highway = highway_class.isna()
            highway_missing_count += int(missing_highway.sum())
            codes = np.full(len(edges), -1, dtype='int16')
            for i, value in enumerate(highway_class.tolist()):
                if value is None:
                    continue
                if value not in highway_to_code:
                    highway_to_code[value] = len(highway_to_code)
                codes[i] = highway_to_code[value]

            edge_metrics = pd.DataFrame({
                'row_id': edges['row_id'].to_numpy(dtype='int64'),
                'edge_circuity': circuity,
                'orientation_bin': orientation_bin.astype('int16'),
                'highway_code': codes.astype('int16'),
                'component_id': row_component[edges['row_id'].to_numpy()].astype('int32'),
            }).set_index('row_id')

            # Exact line-polygon intersection: candidates come from the spatial index and lengths come from post-intersection geometry length.
            left = edges[['row_id', 'geometry']]
            cand = gpd.sjoin(left, units[['unit_code', 'geometry']], how='inner', predicate='intersects')
            if cand.empty:
                continue
            edge_candidate_rows += len(cand)
            right_geom = units_geom.reindex(cand['unit_code'].to_numpy()).reset_index(drop=True)
            left_geom = cand.geometry.reset_index(drop=True)
            try:
                clipped = left_geom.intersection(right_geom)
                clip_len = clipped.length.to_numpy(dtype='float64')
            except Exception as exc:
                append_anomaly(anomaly_records, city_id, network_type, 'all', 'geometry_clip_failed',
                               f'edge-unit intersection failed: {type(exc).__name__}: {exc}', severity='error',
                               affected_count=len(cand), source_file=str(gpkg_path))
                raise
            valid_clip = np.isfinite(clip_len) & (clip_len > 0.01)
            edge_clip_zero_count += int((~valid_clip).sum())
            if not valid_clip.any():
                continue
            cand = cand.loc[valid_clip, ['unit_code', 'row_id']].copy()
            clip_len = clip_len[valid_clip]
            m = edge_metrics.reindex(cand['row_id'].to_numpy())
            part = pd.DataFrame({
                'unit_code': cand['unit_code'].to_numpy(dtype='int32'),
                'clip_len_m': clip_len.astype('float32'),
                'edge_circuity': m['edge_circuity'].to_numpy(dtype='float32'),
                'orientation_bin': m['orientation_bin'].to_numpy(dtype='int16'),
                'highway_code': m['highway_code'].to_numpy(dtype='int16'),
                'component_id': m['component_id'].to_numpy(dtype='int32'),
            })
            edge_unit_parts.append(part)

            if (start // EDGE_CHUNK_SIZE) % 5 == 0:
                print(f'  {city_id} {network_type}: processed edges {min(start + max_features, total_features):,}/{total_features:,}, candidates {edge_candidate_rows:,}')

            del edges, cand, left, right_geom, left_geom, part, edge_metrics
            gc.collect()

        edge_units = pd.concat(edge_unit_parts, ignore_index=True) if edge_unit_parts else pd.DataFrame({
            'unit_code': pd.Series(dtype='int32'),
            'clip_len_m': pd.Series(dtype='float32'),
            'edge_circuity': pd.Series(dtype='float32'),
            'orientation_bin': pd.Series(dtype='int16'),
            'highway_code': pd.Series(dtype='int16'),
            'component_id': pd.Series(dtype='int32'),
        })
        log['seconds_edge_overlay'] = round(time.time() - overlay_start, 3)
        log['edge_candidate_rows'] = int(edge_candidate_rows)
        log['edge_clip_zero_count'] = int(edge_clip_zero_count)
        log['circuity_invalid_count'] = int(circuity_invalid_count)
        log['orientation_invalid_count'] = int(orientation_invalid_count)
        log['highway_missing_count'] = int(highway_missing_count)

        if edge_clip_zero_count:
            append_anomaly(anomaly_records, city_id, network_type, 'all', 'zero_length_clipped_edges',
                           'Some edges only touched units at points or had near-zero clipped length and were skipped', affected_count=int(edge_clip_zero_count),
                           source_file=str(gpkg_path))
        if circuity_invalid_count:
            append_anomaly(anomaly_records, city_id, network_type, 'all', 'circuity_denominator_zero_or_invalid',
                           'Some edge endpoint straight-line distances were zero/missing; edge_circuity was set to NaN', affected_count=int(circuity_invalid_count),
                           source_file=str(gpkg_path))
        if orientation_invalid_count:
            append_anomaly(anomaly_records, city_id, network_type, 'all', 'orientation_unavailable',
                           'Some edge orientations could not be computed from endpoint coordinates', affected_count=int(orientation_invalid_count),
                           source_file=str(gpkg_path))
        if highway_missing_count:
            append_anomaly(anomaly_records, city_id, network_type, 'all', 'highway_missing',
                           'Some edges are missing the highway field and are skipped for road_hierarchy_entropy calculation',
                           affected_count=int(highway_missing_count), source_file=str(gpkg_path))

        metrics = build_metrics_table(units, nodes, edge_units, city_info, network_type, anomaly_records)
        city_metrics = metrics[metrics['scale'].isin(CITY_SCALES)].copy()
        local_metrics = metrics[metrics['scale'].isin(LOCAL_SCALES)].copy()

        safe_to_parquet(city_metrics, paths['city'])
        safe_to_parquet(local_metrics, paths['local'])
        safe_to_csv(pd.DataFrame(anomaly_records), paths['anomaly'])

        log['city_metric_rows'] = len(city_metrics)
        log['local_metric_rows'] = len(local_metrics)
        log['status'] = 'success'
        log['seconds_total'] = round(time.time() - t_all, 3)
        log_df = pd.DataFrame([log])
        safe_to_csv(log_df, paths['log'])
        print(f'[{now_iso()}] Completed {city_id} {network_type}: city_rows={len(city_metrics)}, local_rows={len(local_metrics)}, seconds={log["seconds_total"]}')

        del units, nodes, edge_units, metrics, city_metrics, local_metrics, edge_unit_parts
        gc.collect()

    except Exception as exc:
        log['status'] = 'failed'
        log['message'] = f'{type(exc).__name__}: {exc}'
        log['seconds_total'] = round(time.time() - t_all, 3)
        append_anomaly(anomaly_records, city_id, network_type, 'all', 'processing_failed',
                       log['message'], severity='error', source_file=str(gpkg_path), sample_value=traceback.format_exc()[-2000:])
        safe_to_csv(pd.DataFrame(anomaly_records), paths['anomaly'])
        safe_to_csv(pd.DataFrame([log]), paths['log'])
        print(f'[{now_iso()}] Failed {city_id} {network_type}: {log["message"]}')
        raise


In [ ]:
# ==== 7. Run per-city calculation ====

cities_to_run = parse_city_list(city_master)
networks_to_run = parse_network_list()
print('Cities to process:', len(cities_to_run), 'Networks to process:', networks_to_run)

city_info_map = city_master.set_index('city_id')
run_start = time.time()
for city_id in cities_to_run:
    for network_type in networks_to_run:
        process_city_network(city_id, network_type, city_info_map.loc[city_id])

print('Per-city calculation loop finished, elapsed seconds:', round(time.time() - run_start, 3))


In [ ]:
# ==== 8. Assemble final tables, anomaly table, logs, and quality checks ====

def collect_expected_chunks(cities: list[str], networks: list[str]) -> tuple[list[Path], list[Path], list[Path], list[Path], list[dict]]:
    """Collect expected chunk paths and list missing items."""
    city_paths, local_paths, anomaly_paths, log_paths, missing = [], [], [], [], []
    for city_id in cities:
        for network_type in networks:
            p = chunk_paths(city_id, network_type)
            if p['city'].exists():
                city_paths.append(p['city'])
            else:
                missing.append({'city_id': city_id, 'network_type': network_type, 'missing': 'city_chunk'})
            if p['local'].exists():
                local_paths.append(p['local'])
            else:
                missing.append({'city_id': city_id, 'network_type': network_type, 'missing': 'local_chunk'})
            if p['anomaly'].exists():
                anomaly_paths.append(p['anomaly'])
            if p['log'].exists():
                log_paths.append(p['log'])
            else:
                missing.append({'city_id': city_id, 'network_type': network_type, 'missing': 'log_chunk'})
    return city_paths, local_paths, anomaly_paths, log_paths, missing


def build_quality_check(city_df: pd.DataFrame, local_df: pd.DataFrame, anomaly_df: pd.DataFrame) -> pd.DataFrame:
    """Generate city/network/scale-level quality checks."""
    all_df = pd.concat([city_df, local_df], ignore_index=True)
    key_missing_cols = [
        'edge_density_km_per_km2', 'node_density_per_km2', 'intersection_density_per_km2',
        'edge_circuity_mean', 'orientation_entropy', 'road_hierarchy_entropy', 'betweenness_gini'
    ]
    rows = []
    anomaly_counts = pd.DataFrame()
    if not anomaly_df.empty and {'city_id', 'network_type', 'scale'}.issubset(anomaly_df.columns):
        anomaly_counts = anomaly_df.groupby(['city_id', 'network_type', 'scale']).size().rename('anomaly_record_count')
    for (city_id, network_type, scale), sub in all_df.groupby(['city_id', 'network_type', 'scale'], sort=True):
        has_area_km2 = 'area_km2' in sub.columns
        area_match = False
        if has_area_km2 and 'unit_area_km2' in sub.columns and len(sub):
            area_match = bool(np.allclose(
                pd.to_numeric(sub['area_km2'], errors='coerce'),
                pd.to_numeric(sub['unit_area_km2'], errors='coerce'),
                equal_nan=True,
            ))
        row = {
            'city_id': city_id,
            'network_type': network_type,
            'scale': scale,
            'row_count': len(sub),
            'valid_count': int(sub['valid_metric'].sum()),
            'invalid_count': int((~sub['valid_metric']).sum()),
            'valid_ratio': float(sub['valid_metric'].mean()) if len(sub) else np.nan,
            'has_area_km2': bool(has_area_km2),
            'area_km2_missing_rate': float(sub['area_km2'].isna().mean()) if has_area_km2 and len(sub) else np.nan,
            'area_km2_matches_unit_area_km2': area_match,
            'total_edge_length_km_sum': float(sub['total_edge_length_km'].sum(skipna=True)),
            'edge_density_mean': float(sub['edge_density_km_per_km2'].mean(skipna=True)),
            'node_density_mean': float(sub['node_density_per_km2'].mean(skipna=True)),
            'anomaly_record_count': 0,
        }
        for col in key_missing_cols:
            row[f'{col}_missing_rate'] = float(sub[col].isna().mean()) if col in sub.columns and len(sub) else np.nan
        for anomaly_scale in [scale, 'all', 'graph', 'gpkg', 'graphml']:
            key = (city_id, network_type, anomaly_scale)
            if not anomaly_counts.empty and key in anomaly_counts.index:
                row['anomaly_record_count'] += int(anomaly_counts.loc[key])
        rows.append(row)
    qc = pd.DataFrame(rows)
    return qc.sort_values(['city_id', 'network_type', 'scale']).reset_index(drop=True)

all_cities = expected_cities if not CITY_FILTER else cities_to_run
all_networks = NETWORK_TYPES if not NETWORK_FILTER else networks_to_run
city_paths, local_paths, anomaly_paths, log_paths, missing = collect_expected_chunks(all_cities, all_networks)

if missing:
    print('Missing chunks remain; final tables are not written yet. Missing examples:')
    print(pd.DataFrame(missing).head(20).to_string(index=False))
elif not WRITE_FINAL:
    print('STEP06_WRITE_FINAL=0; chunks are complete but final table assembly was skipped.')
else:
    print('Starting final output assembly...')
    city_df = pd.concat([pd.read_parquet(p) for p in city_paths], ignore_index=True)
    local_df = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
    # Backward-compatible resume for old chunks: if temporary chunks were generated before the field was added, also fill area_km2 in final tables.
    for df in [city_df, local_df]:
        if 'area_km2' not in df.columns and 'unit_area_km2' in df.columns:
            insert_at = list(df.columns).index('unit_area_km2') + 1
            df.insert(insert_at, 'area_km2', df['unit_area_km2'])
        elif 'area_km2' in df.columns and 'unit_area_km2' in df.columns:
            df['area_km2'] = df['area_km2'].fillna(df['unit_area_km2'])
    anomaly_frames = []
    if preflight_anomaly_path.exists():
        try:
            anomaly_frames.append(pd.read_csv(preflight_anomaly_path))
        except Exception:
            pass
    for p in anomaly_paths:
        try:
            anomaly_frames.append(pd.read_csv(p))
        except pd.errors.EmptyDataError:
            pass
    anomaly_df = pd.concat(anomaly_frames, ignore_index=True) if anomaly_frames else pd.DataFrame(columns=[
        'timestamp', 'city_id', 'network_type', 'scale', 'unit_id', 'severity', 'anomaly_type',
        'message', 'affected_count', 'sample_value', 'source_file'
    ])
    log_df = pd.concat([pd.read_csv(p) for p in log_paths], ignore_index=True)

    # Sort rows so CSV/Parquet outputs are stable for diffing.
    city_df = city_df.sort_values(['city_id', 'network_type', 'scale', 'unit_id']).reset_index(drop=True)
    local_df = local_df.sort_values(['city_id', 'network_type', 'scale', 'unit_id']).reset_index(drop=True)
    anomaly_df = anomaly_df.sort_values(['city_id', 'network_type', 'scale', 'anomaly_type'], na_position='last').reset_index(drop=True)
    log_df = log_df.sort_values(['city_id', 'network_type']).reset_index(drop=True)

    qc_df = build_quality_check(city_df, local_df, anomaly_df)

    safe_to_csv(city_df, OUTPUT_CITY_CSV)
    safe_to_parquet(city_df, OUTPUT_CITY_PARQUET)
    safe_to_csv(local_df, OUTPUT_LOCAL_CSV)
    safe_to_parquet(local_df, OUTPUT_LOCAL_PARQUET)
    safe_to_csv(anomaly_df, OUTPUT_ANOMALY_CSV)
    safe_to_csv(log_df, OUTPUT_LOG_CSV)
    safe_to_csv(qc_df, OUTPUT_QC_CSV)

    print('Final outputs complete:')
    print('city_morphology_metrics:', city_df.shape)
    print(city_df.groupby(['network_type', 'scale']).size().to_string())
    print('local_morphology_metrics:', local_df.shape)
    print(local_df.groupby(['network_type', 'scale']).size().to_string())
    print('anomaly_records:', anomaly_df.shape)
    if not anomaly_df.empty and 'anomaly_type' in anomaly_df.columns:
        print(anomaly_df['anomaly_type'].value_counts().head(20).to_string())
    print('quality_checks:', qc_df.shape)
    print(qc_df.groupby('scale')[['row_count', 'valid_count', 'invalid_count']].sum().to_string())
